### Allscripts Sunrise (SCM) Drug Catalog Probe Short

Run this short notebook when we only need the minimum evidence to decide whether SCM drug mapping can be fixed from tables already in catalog.
The outputs from these cells are enough to decide whether `sxammgenericitem`, `sxammproduct`, or `sxammproductpackage` can unblock `drug_exposure`.

In [ ]:
%sql
SELECT table_schema, table_name, column_name, data_type
FROM _exponent.information_schema.columns
WHERE table_name IN (
  'dbo_cv3medicationextension',
  'dbo_sxammgenericitem',
  'dbo_sxammproduct',
  'dbo_sxammproductpackage'
)
ORDER BY table_name, ordinal_position;

In [ ]:
%sql
SELECT *
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem
LIMIT 50;

In [ ]:
%sql
SELECT *
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct
LIMIT 50;

In [ ]:
%sql
SELECT *
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage
LIMIT 50;

In [ ]:
%sql
SELECT
  COUNT(*) AS medext_rows,
  SUM(CASE WHEN PrescriptionGenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS with_generic_item_id,
  COUNT(DISTINCT PrescriptionGenericItemID) AS distinct_generic_item_ids
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN gi.DrugID IS NOT NULL THEN 1 ELSE 0 END) AS generic_item_matches,
  SUM(CASE WHEN gi.RxNormCode IS NOT NULL AND TRIM(CAST(gi.RxNormCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS generic_item_rxnorm_matches
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON CAST(medext.PrescriptionGenericItemID AS STRING) = CAST(gi.DrugID AS STRING);

In [ ]:
%sql
SELECT
  medext.PrescriptionGenericItemID,
  COUNT(*) AS order_rows,
  MAX(gi.RxNormCode) AS sample_rxnorm,
  MAX(gi.Build) AS sample_build
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON CAST(medext.PrescriptionGenericItemID AS STRING) = CAST(gi.DrugID AS STRING)
GROUP BY medext.PrescriptionGenericItemID
ORDER BY order_rows DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  gi.RxNormCode,
  COUNT(*) AS generic_item_rows,
  MAX(rxnorm_source.concept_id) AS rxnorm_source_concept_id,
  MAX(rxnorm_standard.concept_id) AS mapped_standard_concept_id
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
LEFT JOIN _exponent.omop.concept rxnorm_source
  ON rxnorm_source.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
 AND rxnorm_source.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
 AND rxnorm_source.domain_id = 'Drug'
 AND rxnorm_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship maps_to
  ON maps_to.concept_id_1 = rxnorm_source.concept_id
 AND maps_to.relationship_id = 'Maps to'
 AND maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept rxnorm_standard
  ON rxnorm_standard.concept_id = maps_to.concept_id_2
 AND rxnorm_standard.standard_concept = 'S'
 AND rxnorm_standard.domain_id = 'Drug'
 AND rxnorm_standard.invalid_reason IS NULL
WHERE gi.RxNormCode IS NOT NULL
  AND TRIM(CAST(gi.RxNormCode AS STRING)) <> ''
GROUP BY gi.RxNormCode
ORDER BY generic_item_rows DESC
LIMIT 100;